# DomainNet-126 — HEAT integration **smoke test** (one narrow slice)

This is **not** a full experiment. It proves the pipeline works end-to-end on ONE slice —
**arch=resnet50, source=real, target=clipart, η=1e-3** — before investing in the full
domain×η sweep. All real logic is in version-controlled scripts; this notebook only fetches
data/checkpoints and calls `scripts/run_domainnet_smoke.py`, which reuses the existing
adaptation loop + diagnostics (run_tier2.run_p9) and the unchanged ‖ḡ‖/collapse code.

**PASS iff all four checks are sane** (printed itemized, like the WRN gate):
1. AdaContrast checkpoint loads into the wrapper (strict, or exact mismatched keys);
2. source (no-adapt) clipart acc in a plausible band (~0.35–0.55);
3. ‖ḡ‖ finite & sane (proves HEAT's stage taps work on ResNet-50);
4. the collapse criterion triggers across the coarse p-grid (≥1 stable AND ≥1 collapse).

Scope: this slice only. If any check fails, halt and fix integration before scaling.

## 1. GPU check

In [ ]:
import subprocess
o = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(o.stdout if o.returncode == 0 else "⚠️ No GPU — set Runtime → GPU (L4/A100). ~30–45 min on L4.")

## 2. Config — `# === EDIT ME ===`

In [ ]:
# === EDIT ME ===========================================================
REPO_URL   = "https://github.com/octadion/heat.git"
REPO_DIR   = "heat"
GIT_BRANCH = "formulation"   # branch holding the DomainNet files (push there first!)
USE_DRIVE  = True
DRIVE_DIR  = "/content/drive/MyDrive/pstar_results"      # results/JSON
# DomainNet root (images + list .txt land here). Drive-backed = survives disconnect,
# but images are large; /content (ephemeral) is faster if you will finish in one session.
DOMAINNET_ROOT = "/content/domainnet-126"
DOWNLOAD_REAL  = False   # smoke needs ONLY clipart images + the real checkpoint.
                         # real.zip (~5.6 GB) is NOT read by this slice — leave False.

TARGET   = "clipart"
HEAT_LR  = 1e-3
P_GRID   = [0.0, 0.005, 0.02, 0.05]
SEED     = 42
BATCH_SIZE, NUM_WORKERS = 64, 2

# Resolved source URLs (verified — do not guess):
CKPT_DRIVE_FOLDER = "https://drive.google.com/drive/folders/16vTNNzzAt4M1mmeLsOxSFDRzBogaNkJw"
LIST_URL = "https://raw.githubusercontent.com/DianCh/AdaContrast/master/datasets/domainnet-126/{domain}_list.txt"
IMG_URL  = {  # DomainNet "cleaned version" (ai.bu.edu/M3SDA). NOTE: real is NOT under groundtruth/.
    "clipart": "http://csr.bu.edu/ftp/visda/2019/multi-source/groundtruth/clipart.zip",
    "real":    "http://csr.bu.edu/ftp/visda/2019/multi-source/real.zip",
}
# =======================================================================
RESULTS_DIR = DRIVE_DIR if USE_DRIVE else "/content/pstar_results"
print("DOMAINNET_ROOT =", DOMAINNET_ROOT, "| TARGET =", TARGET)

## 3. Mount Drive (optional) + clone/refresh repo

In [ ]:
import os, subprocess
if USE_DRIVE:
    from google.colab import drive; drive.mount("/content/drive")
os.makedirs(RESULTS_DIR, exist_ok=True)
if not os.path.isdir(REPO_DIR):
    clone = ["git","clone"] + (["-b",GIT_BRANCH] if GIT_BRANCH else []) + [REPO_URL,REPO_DIR]
    subprocess.run(clone, check=True)
else:
    if GIT_BRANCH:
        subprocess.run(["git","-C",REPO_DIR,"checkout",GIT_BRANCH], check=False)
    subprocess.run(["git","-C",REPO_DIR,"pull"], check=False)   # refresh scripts
print("branch:", subprocess.run(["git","-C",REPO_DIR,"rev-parse","--abbrev-ref","HEAD"],
                                 capture_output=True, text=True).stdout.strip())
os.chdir("/content/"+REPO_DIR if not os.path.isabs(REPO_DIR) else REPO_DIR)
subprocess.run(["pip","install","-q","-r","requirements.txt"], check=False)
subprocess.run(["pip","install","-q","gdown"], check=False)
os.environ["PYTHONPATH"] = os.getcwd() + os.pathsep + os.environ.get("PYTHONPATH","")
print("cwd =", os.getcwd())

## 4. Resolved download URLs + sizes — **confirm before the large downloads**

Per the integration plan: print the resolved URLs and approximate sizes, and confirm that
**clipart images + the real checkpoint** are sufficient for this slice (real images are
not read).

In [ ]:
print("RESOLVED SOURCES FOR THIS SLICE (source=real model -> adapt clipart):\n")
print("  [required] checkpoint folder (AdaContrast, ~1.1 GB for all 12; we use best_real_2020.pth):")
print("            ", CKPT_DRIVE_FOLDER)
print("  [required] clipart image list (~few MB):", LIST_URL.format(domain="clipart"))
print("  [required] clipart images  (~1 GB, ~48k imgs):", IMG_URL["clipart"])
print("  [optional] real images     (~5.6 GB) — NOT read by this slice; DOWNLOAD_REAL =", DOWNLOAD_REAL)
print("\nTotal required ≈ ~2 GB (clipart.zip + checkpoint). real.zip skipped unless DOWNLOAD_REAL=True.")
print("Checks use ONLY clipart accuracy, so real images are not needed for the smoke.")

## 5. Get image lists (real + clipart) and images (clipart required)

In [ ]:
import os, subprocess
os.makedirs(DOMAINNET_ROOT, exist_ok=True)
for domain in (["clipart","real"] if DOWNLOAD_REAL else ["clipart"]):
    dst = os.path.join(DOMAINNET_ROOT, f"{domain}_list.txt")
    if not os.path.exists(dst):
        subprocess.run(["wget","-q","-O",dst, LIST_URL.format(domain=domain)], check=True)
    print("[list]", dst, "lines:", sum(1 for _ in open(dst)))

def fetch_domain_images(domain):
    marker = os.path.join(DOMAINNET_ROOT, domain)
    if os.path.isdir(marker):
        print(f"[skip] {domain}/ already present."); return
    zp = os.path.join(DOMAINNET_ROOT, f"{domain}.zip")
    print(f"[wget] {IMG_URL[domain]}")
    subprocess.run(["wget","-q","-O",zp, IMG_URL[domain]], check=True)
    print(f"[unzip] {zp} -> {DOMAINNET_ROOT}")
    subprocess.run(["unzip","-q",zp,"-d",DOMAINNET_ROOT], check=True)
    os.remove(zp)

fetch_domain_images("clipart")
if DOWNLOAD_REAL: fetch_domain_images("real")
print("[ok] images ready under", DOMAINNET_ROOT)

## 6. Get the source=real checkpoint (AdaContrast)
Downloads the source-model folder with `gdown` and locates `best_real_2020.pth` (pattern
`*real*2020*.pth`). Echoes the resolved path.

In [ ]:
import os, glob, subprocess
CKPT_DIR = "experiments/checkpoints/domainnet126_source"
os.makedirs(CKPT_DIR, exist_ok=True)
found = glob.glob(os.path.join(CKPT_DIR, "**", "*real*2020*.pth"), recursive=True)
if not found:
    subprocess.run(["gdown","--folder",CKPT_DRIVE_FOLDER,"-O",CKPT_DIR], check=True)
    found = glob.glob(os.path.join(CKPT_DIR, "**", "*real*2020*.pth"), recursive=True)
assert found, f"no *real*2020*.pth found under {CKPT_DIR} — inspect the folder contents."
CKPT_REAL = found[0]
print("CKPT_REAL =", CKPT_REAL)

## 7. Run the smoke test — 4-check PASS/FAIL
Reuses `run_tier2.run_p9` (adaptation loop + diagnostics) + the unchanged ‖ḡ‖/collapse
code. Source acc over clipart, then HEAT over the coarse p-grid; prints the itemized
verdict and writes `RESULTS_DIR/domainnet_smoke/domainnet_smoke_clipart.json`.

In [ ]:
import subprocess, os, json
out_dir = os.path.join(RESULTS_DIR, "domainnet_smoke")
cmd = [
    "python","scripts/run_domainnet_smoke.py",
    "--domainnet-root", DOMAINNET_ROOT, "--ckpt", CKPT_REAL,
    "--target", TARGET, "--heat-lr", repr(HEAT_LR),
    "--p-grid", *[repr(p) for p in P_GRID],
    "--batch-size", str(BATCH_SIZE), "--num-workers", str(NUM_WORKERS),
    "--seed", str(SEED), "--out-dir", out_dir,
]
print("$", " ".join(cmd), "\n")
rc = subprocess.run(cmd).returncode
print("\nsmoke exit code:", rc, "(0 = PASS, 1 = FAIL)")